In [ ]:
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

In [ ]:
# data = pd.read_csv('/content/drive/MyDrive/Data/prepared data/autoencoder_dataset.csv')
data = pd.read_csv('/content/drive/MyDrive/Data/prepared data/de_autoencoder_comb_dataset.csv')
# data = pd.read_csv('/content/drive/MyDrive/Data/prepared data/de_autoencoder_mask_dataset.csv')
# data = pd.read_csv('/content/drive/MyDrive/Data/prepared data/de_autoencoder_gauss_dataset.csv')

In [ ]:
print(data.columns)

Index(['volunteer', 'device', 'kept_paths', 'mu', 'sigma', 'feat_0', 'feat_1',
       'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_8',
       'feat_9', 'feat_10', 'feat_11', 'feat_12', 'feat_13', 'feat_14',
       'feat_15', 'Age', 'Gender', 'Height[m]', 'Weight[kg]', 'BMI',
       'HGS_Left', 'HGS_Right', 'Skin[mm]', 'Fat[mm]', 'Rfcsa[cm2]',
       'Volunteer', 'Device', 'Frequency', 'Magnitude', 'Bandstop Frequency',
       'Bandstop Magnitude', 'f_left', 'f_right', 'BW', 'Threshold',
       'Phase_1GHz_deg', 'Phase_2GHz_deg', 'Slope'],
      dtype='object')


In [ ]:
X = data[[
    # signal normalization
    'mu', 'sigma',

    # autoencoder
    'feat_0','feat_1','feat_2','feat_3','feat_4','feat_5',
    'feat_6','feat_7','feat_8','feat_9','feat_10','feat_11',
    'feat_12','feat_13','feat_14','feat_15',

    # resonance features
    'Bandstop Frequency',
    'Bandstop Magnitude',
    'BW',
    'Phase_1GHz_deg',
    'Phase_2GHz_deg',
    'Slope',

    # demographics
    'Age','Gender','Height[m]','Weight[kg]','BMI',

    # device
    'device'
]].copy()

In [ ]:
X['Gender'] = X['Gender'].map({'M':1,'F':0})
X['device'] = X['device'].map({'CopperMountain':0,'nanoVNA':1})
X = X.dropna()

#multiple model approach
y1 = data['Rfcsa[cm2]'].loc[X.index]
y2 = data['Skin[mm]'].loc[X.index]
y3 = data['Fat[mm]'].loc[X.index]

#multi-output regression
y_train = pd.DataFrame({
    'Rfcsa[cm2]': y1,
    'Skin[mm]': y2,
    'Fat[mm]': y3
})

In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y1, test_size=0.2, random_state=42
)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y2, test_size=0.2, random_state=42
)

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X, y3, test_size=0.2, random_state=42
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_train, test_size=0.2, random_state=42
)

In [ ]:
##multiple model approach

model_muscle = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=10, epsilon=0.01))
])

model_skin = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=10, epsilon=0.01))
])

model_fat = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=10, epsilon=0.01))
])

model_muscle.fit(X_train1, y_train1)
model_skin.fit(X_train2, y_train2)
model_fat.fit(X_train3, y_train3)

Pipeline(steps=[('scaler', StandardScaler()), ('svr', SVR(C=10, epsilon=0.01))])

In [ ]:
##multiple output regressor

model = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', MultiOutputRegressor(SVR(kernel='rbf', C=10, epsilon=0.01)))
])

model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svr',
                 MultiOutputRegressor(estimator=SVR(C=10, epsilon=0.01)))])

In [ ]:
y_pred = model_muscle.predict(X_test1)

df_y_muscle = pd.DataFrame({'y_test': y_test1, 'y_pred': y_pred, 'error':(y_test1-y_pred)})
combined_df = X_test.join(df_y_muscle)
# df_filtered = combined_df[abs(combined_df['error']) < 1.5].copy()
# df_filtered['Gender'] = df_filtered['Gender'].apply(lambda x: 'M' if x == 1 else 'F')
# df_filtered['device'] = df_filtered['device'].apply(lambda x: 'nanoVNA' if x == 1 else 'CopperMountain')

from sklearn.metrics import mean_absolute_error
mae_muscle = mean_absolute_error(y_test1, y_pred)


In [ ]:
y_pred = model_skin.predict(X_test2)

df_y_skin = pd.DataFrame({'y_test2': y_test2, 'y_pred': y_pred, 'error':(y_test2-y_pred)})
combined_df = X_test.join(df_y_skin)
# df_filtered = combined_df[abs(combined_df['error']) < 1.5].copy()
# df_filtered['Gender'] = df_filtered['Gender'].apply(lambda x: 'M' if x == 1 else 'F')
# df_filtered['device'] = df_filtered['device'].apply(lambda x: 'nanoVNA' if x == 1 else 'CopperMountain')

from sklearn.metrics import mean_absolute_error
mae_skin = mean_absolute_error(y_test2, y_pred)

In [ ]:
y_pred = model_fat.predict(X_test3)

df_y_fat = pd.DataFrame({'y_test': y_test3, 'y_pred': y_pred, 'error':(y_test3-y_pred)})
combined_df = X_test.join(df_y_fat)
# df_filtered = combined_df[abs(combined_df['error']) < 1.5].copy()
# df_filtered['Gender'] = df_filtered['Gender'].apply(lambda x: 'M' if x == 1 else 'F')
# df_filtered['device'] = df_filtered['device'].apply(lambda x: 'nanoVNA' if x == 1 else 'CopperMountain')

from sklearn.metrics import mean_absolute_error
mae_fat = mean_absolute_error(y_test3, y_pred)

In [ ]:
print('MAE Muscle: '+str(mae_muscle))
print('MAE Skin: '+str(mae_skin))
print('MAE Fat: '+str(mae_fat))

MAE Muscle: 0.5033757728715016
MAE Skin: 0.03275275512075846
MAE Fat: 0.981189229380406


In [ ]:
y_pred = model.predict(X_test)

muscle_pred = y_pred[:,0]
skin_pred = y_pred[:,1]
fat_pred = y_pred[:,2]

muscle_label = y_test['Rfcsa[cm2]']
skin_label = y_test['Skin[mm]']
fat_label = y_test['Fat[mm]']

mae_muscle = mean_absolute_error(muscle_label, muscle_pred)
mae_skin = mean_absolute_error(skin_label, skin_pred)
mae_fat = mean_absolute_error(fat_label, fat_pred)

print('MAE Muscle: '+str(mae_muscle))
print('MAE Skin: '+str(mae_skin))
print('MAE Fat: '+str(mae_fat))

MAE Muscle: 0.5033757728715016
MAE Skin: 0.03275275512075846
MAE Fat: 0.981189229380406
